# 05 · Hardening de protocolos: MQTT e OPC UA

**Objetivo:** pegar os dois protocolos que **têm** recursos de segurança e ligá-los de verdade, comparando antes e depois com testes reais.

| | Antes | Depois | Requisitos |
|---|---|---|---|
| MQTT (UNS) | porta 1883, anônimo, texto aberto | 8883 com TLS, contas por cliente, ACL por tópico | SR 1.1, 1.5, 1.7, 2.1, 3.1, 4.1, 4.3 |
| OPC UA | SecurityPolicy None | Basic256Sha256 SignAndEncrypt, certificados confiáveis, login | SR 1.1, 1.2, 1.8, 3.1, 4.1 |

Tudo roda localmente: um `mosquitto` real e um servidor `asyncua` real. A CA privada, os certificados e as senhas são **gerados na hora** em `configs/runtime/` (fora do git). Nenhuma chave vai para o repositório.

### Antes de começar

| | |
|---|---|
| **Para quem** | Quem usa MQTT e OPC UA com a configuração padrão e quer ver o que muda ao ligar a segurança. |
| **Pré-requisitos** | Notebooks 00 a 04. Para executar: mosquitto instalado. Noção de certificado digital ajuda. |
| **Tempo** | 40 minutos |

**Ao final você consegue:**

- Configurar um broker MQTT com TLS, contas por cliente e ACL por tópico.
- Mostrar, numa captura, a diferença entre credencial em claro e criptografada.
- Explicar as três camadas de segurança do OPC UA.
- Dizer por que o cliente também precisa confiar no certificado do servidor.

**Onde isso aparece na fábrica:** A maioria das instalações roda MQTT na porta 1883 sem senha e OPC UA com segurança desligada, porque é o padrão de fábrica. Ligar a segurança é configuração, não compra de produto.

**Uma analogia para guardar:** TLS é o envelope lacrado. Usuário e senha são o crachá. A ACL é a lista de salas em que cada crachá entra. O certificado é o documento com foto que impede alguém de se passar pela recepção.

> **Como estudar:** rode uma célula por vez. Sempre que aparecer **Pare e preveja**, responda mentalmente antes de executar. Errar a previsão é o que faz o conceito fixar.

In [1]:
# Configuração: raiz do repositório no caminho de import, tabelas largas
import sys, pathlib, logging, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
from IPython.display import Image, Markdown, display
from otsec.model import CAPTURES
from otsec import hardening as h
pki = h.pki()
print("PKI do lab:", sorted(p.name for p in pki.iterdir() if p.suffix == ".pem"))
senhas = h.passwords()
print("contas geradas:", list(senhas))

PKI do lab: ['broker-1.pem', 'broker-1_key.pem', 'ca.pem', 'ca_key.pem', 'historian-client.pem', 'historian-client_key.pem', 'scada-opcua.pem', 'scada-opcua_key.pem', 'untrusted-client.pem', 'untrusted-client_key.pem']
contas geradas: ['scada', 'historian', 'dashboard']


## 1. MQTT antes: o que um observador vê

Ligamos o broker inseguro e capturamos o loopback enquanto o SCADA se conecta (a captura precisa de root).

> **Pare e preveja.** O SCADA se conecta ao broker na porta 1883 informando usuário e senha. Capturando os pacotes, a senha aparece legível nos bytes?

In [2]:
with h.Mosquitto(secure=False):
    print("anônimo conecta? ", h.mqtt_try_connect(1883))
    cap = h.sniff_credentials(1883, lambda: h.mqtt_try_connect(1883, user="scada", password="Acme2024!"))
i = cap["blob"].find(b"MQTT")
print(f"{cap['packets']} pacotes capturados. Bytes do pacote CONNECT, como um observador vê:")
print(cap["blob"][i:i + 60])
print("senha visível?", b"Acme2024!" in cap["blob"])

anônimo conecta?  connected


22 pacotes capturados. Bytes do pacote CONNECT, como um observador vê:
b'MQTT\x04\xc2\x00\x05\x00\x00\x00\x05scada\x00\tAcme2024!\x10\x1e\x00\x04MQTT\x04\xc2\x00\x05\x00\x00\x00\x05scada\x00\tAcme2024!'
senha visível? True


Qualquer cliente entra sem senha, e quando alguém usa senha ela passa **em claro** (`Acme2024!` aparece nos bytes capturados).

## 2. MQTT depois: TLS, contas e ACL

In [3]:
print(h.ACL)

# one account per client, least privilege
user scada
topic write acme/sao-paulo/packaging/#

user historian
topic read acme/#

user dashboard
topic read acme/sao-paulo/packaging/+/+/state



> **Pare e preveja.** No broker endurecido, o SCADA de São Paulo tenta publicar um estado na árvore de tópicos de Curitiba. O broker devolve erro, ou aceita e descarta em silêncio?

In [4]:
S = {"user": "scada", "password": senhas["scada"]}
H = {"user": "historian", "password": senhas["historian"]}
with h.Mosquitto(secure=True):
    testes = {
        "anônimo com TLS": h.mqtt_try_connect(8883, tls=True),
        "senha certa, sem TLS": h.mqtt_try_connect(8883, **S),
        "TLS sem confiar na CA da planta": h.mqtt_try_connect(8883, **S, tls=True, ca=False),
        "senha errada": h.mqtt_try_connect(8883, user="scada", password="Acme2024!", tls=True),
        "scada, senha certa, TLS": h.mqtt_try_connect(8883, **S, tls=True),
    }
    acl = {
        "scada publica estado da linha 1 (permitido)": h.mqtt_pubsub(8883, S, H, "acme/sao-paulo/packaging/line-1/filler-01/state", tls=True),
        "scada publica estado de Curitiba (fora da sua árvore)": h.mqtt_pubsub(8883, S, H, "acme/curitiba/packaging/line-3/filler-01/state", tls=True),
        "historian tenta publicar (só tem leitura)": h.mqtt_pubsub(8883, H, H, "acme/sao-paulo/packaging/line-1/filler-01/state", tls=True),
    }
    cap = h.sniff_credentials(8883, lambda: h.mqtt_try_connect(8883, **S, tls=True))
display(pd.DataFrame(testes.items(), columns=["tentativa", "resultado"]))
display(pd.DataFrame(acl.items(), columns=["publicação", "chegou ao assinante?"]))
print(f"captura TLS: {cap['packets']} pacotes, senha visível nos bytes? {senhas['scada'].encode() in cap['blob']}")

,tentativa,resultado
0,anônimo com TLS,refused: Not authorized
1,"senha certa, sem TLS",refused: no CONNACK
2,TLS sem confiar na CA da planta,refused: SSLCertVerificationError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify...
3,senha errada,refused: Not authorized
4,"scada, senha certa, TLS",connected


,publicação,chegou ao assinante?
0,scada publica estado da linha 1 (permitido),True
1,scada publica estado de Curitiba (fora da sua árvore),False
2,historian tenta publicar (só tem leitura),False


captura TLS: 38 pacotes, senha visível nos bytes? False


**Leitura**

* Sem conta não entra; sem TLS não completa a conexão; sem confiar na **CA privada da planta** o cliente recusa o broker, o que protege contra um broker falso (**T0830 Adversary-in-the-Middle**).
* A ACL impede que o SCADA de São Paulo publique na árvore de Curitiba e que o historiador publique qualquer coisa. Isso fecha **T0856 Spoof Reporting Message**: ninguém injeta estados falsos de máquina no UNS.
* Na captura TLS a senha **não aparece**.

## 3. Política de senhas

A senha antiga era curta, tinha palavra comum e ano. As contas novas são geradas aleatoriamente e passam pela mesma checagem.

In [5]:
pd.DataFrame([{"senha": "Acme2024!", "problemas": h.password_issues("Acme2024!", "scada")}] +
             [{"senha": f"(gerada para {u}, {len(p)} caracteres)", "problemas": h.password_issues(p, u) or "nenhum"} for u, p in senhas.items()])

,senha,problemas
0,Acme2024!,"[shorter than 12 characters, contains a common word, contains a year]"
1,"(gerada para scada, 24 caracteres)",nenhum
2,"(gerada para historian, 24 caracteres)",nenhum
3,"(gerada para dashboard, 24 caracteres)",nenhum


## 4. OPC UA: só sessões assinadas, criptografadas e de clientes confiáveis

O servidor do SCADA aceita apenas **Basic256Sha256 SignAndEncrypt**, só confia em certificados emitidos pela CA da planta e exige usuário e senha. Testamos quatro clientes.

> **Pare e preveja.** Um cliente OPC UA chega com um certificado válido, mas autoassinado e desconhecido da planta. O servidor deve aceitar a sessão?

In [6]:
async def bateria():
    srv = h.opcua_server(secure=True); await h.opcua_start(srv, secure=True)
    try:
        return {
            "cliente sem segurança (None)": await h.opcua_try(secure=False),
            "certificado autoassinado desconhecido": await h.opcua_try("historian", senhas["historian"], cert="untrusted-client"),
            "certificado confiável, sem usuário": await h.opcua_try(None, None),
            "certificado confiável, senha errada": await h.opcua_try("historian", "errada"),
            "certificado confiável e login correto": await h.opcua_try("historian", senhas["historian"]),
        }
    finally:
        await srv.stop()

pd.DataFrame((await bateria()).items(), columns=["cliente", "resultado"])

self-signed certificate


,cliente,resultado
0,cliente sem segurança (None),"refused: UaError: No matching endpoints: 1, http://opcfoundation.org/UA/SecurityPolicy..."
1,certificado autoassinado desconhecido,refused: BadCertificateUntrusted: The certificate is not trusted.(BadCertificateUntrus...
2,"certificado confiável, sem usuário",refused: BadIdentityTokenRejected: The user identity token is valid but the server has...
3,"certificado confiável, senha errada",refused: BadUserAccessDenied: User does not have permission to perform the requested o...
4,certificado confiável e login correto,"connected, Speed = 468.0"


**Leitura:** o OPC UA tem três camadas de segurança, e as três estão ativas aqui:

1. **Canal seguro:** sem `SignAndEncrypt` não há nem endpoint compatível (`No matching endpoints`).
2. **Identidade da aplicação (SR 1.2):** o certificado do cliente precisa ser emitido pela CA da planta. Um certificado autoassinado qualquer é recusado com `BadCertificateUntrusted`.
3. **Identidade do usuário (SR 1.1):** mesmo com certificado válido, sem login correto a sessão é rejeitada.

## 5. E o Modbus?

Modbus/TCP clássico não tem nada disso. Existe o **Modbus/TCP Security** (TLS na porta 802), mas poucos equipamentos suportam. Por isso, na célula, a proteção vem da arquitetura: firewall (04) e IDS (06).

## Resumo

* Hardening é **configuração**, não produto novo: TLS, contas, ACL e certificados já existiam no mosquitto e no OPC UA.
* Cada afirmação deste notebook é um teste em `tests/test_hardening.py` e conta como evidência no exercício 07.
* Segredos gerados em tempo de execução, nunca commitados: é o mínimo esperado de um repositório público.

---
## Erros comuns

- **TLS sem verificar o certificado do servidor.** O cliente cifra a conversa com qualquer um, inclusive um broker falso.
- **Uma conta para todos os clientes.** Não dá para saber quem publicou nem revogar um acesso sem parar todos.
- **Senha ou chave no repositório.** Segredos são gerados na implantação e guardados num cofre, nunca no git.
- **Segurança "None" no OPC UA em produção.** Qualquer cliente lê e escreve. Aceite só políticas assinadas e criptografadas.

## Teste seu entendimento

Responda antes de abrir cada resposta.

**1. Por que o cliente que não confia na CA da planta se recusa a conectar, mesmo com a senha certa?**

<details><summary>Ver resposta</summary>

Porque ele não consegue confirmar que o servidor é quem diz ser. Recusar evita entregar a senha a um broker falso.

</details>

**2. O que a ACL por tópico impede que usuário e senha sozinhos não impedem?**

<details><summary>Ver resposta</summary>

Que um cliente autenticado publique ou leia fora da sua área, por exemplo injetando estados falsos de outra planta.

</details>

**3. Quais são as três camadas de segurança do OPC UA mostradas aqui?**

<details><summary>Ver resposta</summary>

Canal seguro (assinatura e criptografia), identidade da aplicação (certificado confiável) e identidade do usuário (login).

</details>

## Exercícios para praticar

1. Crie uma conta `dashboard` só de leitura para tópicos de estado e teste que ela não consegue publicar.
2. Gere um certificado de cliente fora da CA da planta e confirme a recusa do servidor OPC UA.
3. Passe três senhas suas (inventadas) pela função de política de senhas e veja quais problemas ela aponta.

## Glossário

| Termo | Significado |
|---|---|
| **TLS** | Protocolo que cifra a comunicação e autentica o servidor. |
| **CA** | Autoridade certificadora, quem assina e dá validade aos certificados. |
| **ACL** | Lista de controle de acesso: o que cada conta pode ler e escrever. |
| **SignAndEncrypt** | Modo do OPC UA em que as mensagens são assinadas e cifradas. |
| **Lista de confiança** | Certificados que o servidor aceita. |
| **Hardening** | Reduzir a exposição de um sistema ajustando a configuração. |

**Próximo passo:** [06 · Detecção de intrusão no Modbus](06_deteccao_de_intrusao.ipynb), porque prevenir não basta: é preciso perceber.